In [169]:

# 1---> Preprocessing And Cleaning
# 2--> Train Test Split
# 3---> Apply BOW ,TF-IDF ,Woed2Vec
# 4----> Apply machine leraning

In [170]:
# Data set laod
import pandas as pd

df = pd.read_csv("Amazon_Reviews.csv", encoding="latin1")
# df

In [171]:
# choosing rating and review test from the entire data sets 
data=df[["Rating","Review Text"]]
# data

In [172]:
# 1---> Preprocessing And Cleaning

# df.head()
data["Rating"].isnull().sum()
data.isnull().sum() 
data=data.dropna()       # handling missing value here
data.isnull().sum
# data["Rating"].value_counts()
# convert  text into numerical form of all the rating data
data["Rating"] = (
    data["Rating"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype("Int64")
)
# Assigned the value with neagative review is 0 and positive review is to be 1
data["Rating"]=data["Rating"].apply(lambda x: 0 if x<3 else 1 
                                   )
data["Rating"].value_counts()                                   

Rating
0    14350
1     6705
Name: count, dtype: int64

In [173]:
data["Rating"].value_counts()                                   

Rating
0    14350
1     6705
Name: count, dtype: int64

In [174]:
# Lowering the cases
data["Review Text"]=data["Review Text"].str.lower()

In [175]:
import nltk
from nltk.corpus import stopwords 
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [176]:
import re
from nltk.corpus import stopwords

# Load stopwords only ONCE
stop_words = set(stopwords.words("english"))

data["Review Text"] = (
    data["Review Text"]
    .fillna("")
    .astype(str)
    
    # Remove HTML tags
    .str.replace(r"<[^>]*>", " ", regex=True)
    
    # Remove URLs
    .str.replace(r"http\S+|www\S+", " ", regex=True)
    
    # Remove special characters
    .str.replace(r"[^a-zA-Z0-9\s]", " ", regex=True)
    
    # Remove extra spaces
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove stopwords
data["Review Text"] = data["Review Text"].apply(
    lambda x: " ".join(word for word in x.split() if word.lower() not in stop_words)
)

In [177]:
data = data.drop_duplicates(subset=["Review Text"], keep="first")
data


,Rating,Review Text
0,0,registered website tried order laptop entered ...
1,0,multiple orders one turned driver phone door n...
2,0,informed reprobates would going visit sick rel...
3,0,bought amazon problems happy service price ama...
4,0,could give lower rate would cancelled amazon p...
...,...,...
21209,1,perfect order fulfillment fast delivery amazon...
21210,1,perfect order fulfillment fast delivery amazon...
21211,1,always find going back amazon becouse prices g...
21212,1,placed abundance orders amazon last couple yea...


In [178]:
# !pip install nltk

In [179]:
# data["Review Text"]
# apply lemmatizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
wnl=WordNetLemmatizer()
def lemmatize_words(text):
    words = word_tokenize(text)
    return " ".join([wnl.lemmatize(word) for word in words])
    

In [180]:
data["Review Text"] =data["Review Text"].apply(lemmatize_words)
# # data


In [181]:
# # Separate the dependent value and independent value
# x=data["Review Text"]
# y=data["Rating"]
# x,y
# # train test split 
# from sklearn.model_selection import train_test_split
# x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)
# # x_train.shape
# x_train

In [182]:
# # Create a bag of words  ----> Text to vector 

# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.naive_bayes import MultinomialNB
# cv = CountVectorizer(
#     max_features=1000,
#     ngram_range=(1, 2)
# )
# X_train = cv.fit_transform(x_train).toarray()
# X_test = cv.transform(x_test).toarray()

In [183]:
# Now use Word 2vec to train the model and use Avg2Vec too
from gensim.models import Word2Vec


# Tokenize text
sentences = data["Review Text"].apply(lambda x: x.lower().split()).tolist()

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [184]:
# sentences
# # method to check the vector by our train model 
# w2v_model.wv["happy"]
# w2v_model.wv.most_similar("happy", topn=10)

In [185]:
# Now time all the indivisual vector into a single list by using Avg2vec

import numpy as np
def avg_word2vec(sentence, model):
    words = sentence.lower().split()
    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)
    
X = np.array([
    avg_word2vec(sentence, w2v_model)
    for sentence in data["Review Text"]
])

X.shape    

(20376, 100)

In [186]:
from sklearn.model_selection import train_test_split
y=data["Rating"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [187]:
# Training Model by using Logistic Regression 
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [188]:
# # this model is use when we use BOW method
# from sklearn.naive_bayes import GaussianNB
# model=GaussianNB()
# model.fit(X_train,y_train)

In [189]:
y_pred=spam_model.predict(X_test)
y_pred,y_test

ValueError: X has 100 features, but GaussianNB is expecting 1000 features as input.

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
confusion_matrix(y_test,y_pred)